<div style="background:linear-gradient(120deg,#0b1f3a 0%,#173f73 55%,#2a7ab8 100%);color:#fff;padding:26px 30px;border-radius:14px">
<div style="font-size:30px;font-weight:700;line-height:1.2">QCC 27.302: the public frontier + one free Toffoli-to-CZ fix in invsqrt</div>
<div style="font-size:15px;margin-top:10px;opacity:.92">Google Verifiable Quantum Circuit Compilation &middot; every file re-verified with the unchanged official verifiers &middot; full credits below</div>
<div style="margin-top:14px"><span style="background:#ffd166;color:#0b1f3a;padding:4px 10px;border-radius:999px;font-weight:700">GM 27.3024+ (local, default salt)</span>
<span style="background:rgba(255,255,255,.18);padding:4px 10px;border-radius:999px;margin-left:6px">invsqrt 78,719.53 (was 78,765.51)</span>
<span style="background:rgba(255,255,255,.18);padding:4px 10px;border-radius:999px;margin-left:6px">built from public sources at run time</span></div>
</div>

This version takes the best public file of each task (2026-11-14) and adds **one exact change of ours to the invsqrt
circuit**: a Toffoli whose target is never read again (it is only measured away) is replaced by measuring first and
applying a classically controlled CZ, which removes one Toffoli on every shot at no qubit cost. Everything else is
other people's work, credited below with links. If this notebook helps you, an upvote is appreciated.

| Task | File | Qubits | Built by (latest change first) |
|---|---|---:|---|
| invsqrt | `dcde7078` **+ our dead-write fix** | 46 | the public 46-qubit line (Syed Asad Ali's own-circuit invsqrt, Ali Haydar Özdağ's `4560ac0c`, as shipped by Alex Chilton and Atakan Aldemir) |
| DQI | `821bf8d1` | 889 | Syed Asad Ali's v12 decoder at extractor cutoff 18 (Ertuğrul Özer) + Areeb Ali's 241 retained products |
| chemistry | npz `2f0cb894` + loaders `93fffb08` | 427 | maoxin123's (8,143,44) table, coded words and 4,038-Toffoli loaders; weight update by Heliosli (and Atakan Aldemir) with prvsiyan's refit code; Lera (marculera)'s table line |

## Acknowledgements
Thank you to **Syed Asad Ali** (from-source invsqrt, DQI and chemistry builders), **Ali Haydar Özdağ** (invsqrt
`4560ac0c`), **Areeb Ali** (product retention), **Ertuğrul Özer** (DQI cutoff), **maoxin123** (chemistry table, coded
words and loaders), **Atakan Aldemir** (weight update and the portfolio this version reads), **prvsiyan** (refit code),
**Lera (marculera)** and the authors her NOTICE credits, **Heliosli**, **Alex Chilton**, **Kirill Tymchishin**,
**WCao24**, **Thomas Gamet**, **manwithacat (James Barlow)**, **SomeJoint**, **busyaprime**, **yoyobar**, **mrmeows**,
**Hosen42**, **Gengsr**, **Ahmed Berat Özer**, the team **i want to run gemma 9b**, **jday96314** and **Oleg Sher**.
And Google Quantum AI and Kaggle for the competition, kickmix and the verifiers.

**What is ours here:** the invsqrt dead-write fix (found with our own circuit-analysis library, proved on all
3,133,452 inputs and every measurement branch, then checked by the official verifier: 78,765.51 -> 78,719.53), and the
[QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) used below for provenance.
Companion notebook: [QCC Toolkit: Simulator & Cost Anatomy](https://www.kaggle.com/code/asteriospanoras/qcc-28-06018-what-works-what-fails-and-why).

<div style="background:#fff7e0;border-left:6px solid #ffb703;padding:12px 16px;border-radius:8px;color:#1b1b1b">
<b>Getting the submission file.</b> This notebook rebuilds and verifies everything when it runs. To get
<code>submission.zip</code>: <b>Copy &amp; Edit</b>, set <code>WRITE_SUBMISSION = True</code> in the next cell and
<b>Run All</b> (about 25 minutes on CPU); the file appears in <code>/kaggle/working</code>. Please keep the credits.
</div>

In [ ]:
WRITE_SUBMISSION = False
import sys, os, pathlib, zipfile, json, time, subprocess, hashlib, math, csv
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
inp = pathlib.Path("/kaggle/input")
KM = next(p for p in inp.rglob("kickmix") if p.is_dir() and (p.parent / "verifiers").is_dir())
VER = KM.parent / "verifiers"; sys.path.insert(0, str(KM))
import kickmix as km
sha = lambda b: hashlib.sha256(b).hexdigest()
SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
src = next(z for z in inp.rglob("submission.zip") if "gap-3-residue" in str(z))
hel = next(z for z in inp.rglob("submission.zip") if "dataset-is-public" in str(z))
PIN = {"invsqrt.kmb": "dcde7078", "dqi.kmb": "821bf8d1", "dfthcsos/submission.npz": "2f0cb894", "dfthcsos/angle_k_0.kmb": "93fffb08"}
with zipfile.ZipFile(src) as Z:
    for n in Z.namelist():
        if n.endswith("/"): continue
        b = Z.read(n)
        if n == "dfthcsos/submission.npz":
            with zipfile.ZipFile(hel) as H: b = H.read(n)
        if n in PIN: assert sha(b).startswith(PIN[n]), (n, sha(b)[:12])
        (SUB / n).parent.mkdir(parents=True, exist_ok=True); (SUB / n).write_bytes(b)
log("public files", {n: sha((SUB / n).read_bytes())[:8] for n in PIN})

## Our change: the invsqrt dead write
Op 3292 of `dcde7078` is `CCX q16 q20 q21`; the next and only use of `q21` is `HMR q21 b61` at op 3403 (measured away).
Measuring first and replacing the Toffoli by `CZ q16 q20 if b61` gives the same values, classical record and phase:
the HMR outcome is uniform either way and the CZ applies exactly the phase `(-1)^(m * q16 * q20)` that the Toffoli's
target would have contributed. One Toffoli fewer per shot.

In [ ]:
c = km.Circuit.from_file(str(SUB / "invsqrt.kmb"), format="kmb"); c.to_file("/tmp/inv.kmx", format="kmx")
lines = open("/tmp/inv.kmx").read().splitlines()
ol = [i for i, l in enumerate(lines) if l.split() and l.split()[0] not in ("APPEND_TO_REGISTER", "REGISTER", "INFO_TAG", "DEBUG_PRINT")]
i1, i2 = ol[3292], ol[3403]
assert lines[i1] == "CCX q16 q20 q21" and lines[i2] == "HMR q21 b61", (lines[i1], lines[i2])
assert not any("b61" in l.split() for l in lines[:i2])
lines = lines[:i1] + ["HMR q21 b61", "CZ q16 q20 if b61"] + lines[i1 + 1:i2] + lines[i2 + 1:]
open("/tmp/inv_fixed.kmx", "w").write("\n".join(lines) + "\n")
km.Circuit.from_file("/tmp/inv_fixed.kmx", format="kmx").to_file(str(SUB / "invsqrt.kmb"), format="kmb")
log("invsqrt fixed", sha((SUB / "invsqrt.kmb").read_bytes())[:12])

## Official verification of the exact bytes that go into `submission.zip`

In [ ]:
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"])
log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
r = {k: REF[k] / res[k]["score"] for k in REF}
print({k: round(x, 6) for k, x in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
OUT_ZIP = "/kaggle/working/submission.zip" if WRITE_SUBMISSION else "/tmp/submission.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 14, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res},
          open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written to", OUT_ZIP)

## Where each file can be found (read from the QCC Circuit Atlas)
The [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) lists every public circuit
file of this competition once, with the public notebooks and datasets that hold the same bytes. The source shown is
one of them (the first in scan order, not necessarily the author: see the credits below for who built each file).
Files newer than the atlas scan (2026-11-12) are marked as such.

In [ ]:
atlas = next((p for p in inp.rglob("circuits.csv") if "atlas" in str(p).lower()), None)
rows = {r["sha12"]: r for r in csv.DictReader(open(atlas))} if atlas else {}
print(f"{'file':28s} {'sha12':12s} {'found in (one of)':58s} {'owner':22s} {'#sources':>8s}")
for p in sorted(SUB.rglob("*")):
    if not p.is_file(): continue
    h = sha(p.read_bytes())[:12]; r = rows.get(h)
    if r: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {r['first_source_ref'][:58]:58s} {r['author'][:22]:22s} {r['n_sources']:>8s}")
    else: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {'(newer than the atlas scan)':58s}")

## Credits (who did what), with links
- **invsqrt** `dcde7078` (as shipped in [Alex Chilton's portfolio](https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio)
  and [Atakan Aldemir's notebooks](https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler)): the
  46-qubit own-circuit line of [Syed Asad Ali](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) and
  [Ali Haydar Özdağ's](https://www.kaggle.com/code/ahoirg/qcc-submit-verify-and-package) `4560ac0c`; its lineage is credited there.
  **Ours:** the dead-write fix above.
- **DQI** `821bf8d1`: [Areeb Ali](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-11042) (241 retained
  products) on Syed Asad Ali's v12 decoder at extractor cutoff 18 ([Ertuğrul Özer](https://www.kaggle.com/code/ozertuu/quantum-ai-27-083-vqcc-dqi-cutoff-17));
  the earlier decoder lineage (team i want to run gemma 9b, Heliosli, SomeJoint, jday96314, Kirill Tymchishin, Atakan Aldemir) is
  credited in those notebooks.
- **Chemistry** npz `2f0cb894` (weights by [Heliosli](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)) + loaders `93fffb08`: [maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-4038-toffoli-coded-word-chemistry)
  ((8,143,44) table, coded words, 4,038-Toffoli loaders); weight update by [Atakan Aldemir](https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler)
  with [prvsiyan's refit code](https://www.kaggle.com/datasets/prvsiyan/vqcc-verified-best-of-circuits-1106a); Lera (marculera)'s
  DF-THC line and the authors her NOTICE credits (Ahmed Berat Özer, kozykappa, anazemcev, mrmeows, michaelrustad, phucthaiv02,
  somejoint, gengsr, qwertykeypadapple, lightningv08, syed3000, Heliosli, busyaprime).
- Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each file keeps its authors' terms (Apache-2.0 where stated);
  the code of this notebook is Apache-2.0.